# Ejercicios: modelos de regresión aplicados a China y a México (desde el año 2000)

**Materia:** Tópicos de Big Data
**Basado en:** `U3_1_modelos_regresion_demografia.ipynb`

En este notebook se repite el procedimiento de la práctica original (regresión lineal, regresión polinomial de grado 2 y Random Forest) para:

1. **Ejercicio 1:** extraer el país **China** y proyectar nacimientos y defunciones.
2. **Ejercicio 2:** extraer **México a partir del año 2000** y proyectar nacimientos y defunciones usando solo esos años para entrenar.

En ambos casos se grafican los resultados de los tres modelos y se comparan contra la proyección de OWID (ONU, escenario medio).

# PARTE 1. Importar las bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor

# PARTE 2. Cargar y preparar el dataset

Mismo procedimiento de la práctica original. Si no hay internet, guarda el CSV como `births-and-deaths-projected-to-2100.csv` en la misma carpeta que este notebook y el código lo detecta automáticamente.

In [ ]:
url = "https://ourworldindata.org/grapher/births-and-deaths-projected-to-2100.csv?v=1&csvType=full&useColumnShortNames=true"

try:
    df = pd.read_csv(
        url,
        storage_options={
            "User-Agent": "Our World In Data data fetch/1.0"
        }
    )
except Exception as error:
    print("No fue posible descargar el archivo desde internet:", error)
    print("Se utilizará el archivo local 'births-and-deaths-projected-to-2100.csv'")
    df = pd.read_csv("births-and-deaths-projected-to-2100.csv")

df.head()

In [ ]:
df = df.rename(columns={
    "entity": "Entity",
    "code": "Code",
    "year": "Year",
    "births__sex_all__age_all__variant_estimates": "births_estimates",
    "births__sex_all__age_all__variant_medium__projected": "births_projected",
    "deaths__sex_all__age_all__variant_estimates": "deaths_estimates",
    "deaths__sex_all__age_all__variant_medium__projected": "deaths_projected",
})

# Columna unificada: dato histórico si existe; si no, la proyección de OWID
df["births"] = df["births_estimates"].fillna(df["births_projected"])
df["deaths"] = df["deaths_estimates"].fillna(df["deaths_projected"])

df.head()

# PARTE 3. Funciones auxiliares

Para no copiar las mismas celdas cuatro veces (China y México, cada uno con nacimientos y defunciones), los pasos de la práctica original se agrupan en tres funciones. **El contenido es el mismo** que en el notebook original:

| Paso de la práctica original | Dónde está en la función |
|---|---|
| `X = df_historico[["Year"]]`, `y = df_historico["births"]` | `proyectar_tres_modelos` |
| `LinearRegression().fit(X, y)` | Modelo 1 |
| `PolynomialFeatures(degree=2)` + `LinearRegression()` | Modelo 2 |
| `RandomForestRegressor(n_estimators=100, random_state=42)` | Modelo 3 |
| Años futuros 2024–2100 como `DataFrame` con columna `Year` | `X_futuro` |
| Gráficas de cada modelo, comparación y comparación contra OWID | `graficar_modelos_individuales` y `graficar_comparacion_owid` |

In [ ]:
# Años para los que generamos nuestras propias proyecciones
anios_futuros = np.arange(2024, 2101).reshape(-1, 1)
X_futuro = pd.DataFrame(anios_futuros, columns=["Year"])

COLORES = {
    "Regresión lineal": "tab:orange",
    "Regresión polinomial": "tab:green",
    "Random Forest": "tab:red",
}


def proyectar_tres_modelos(df_historico, variable):
    """Entrena los 3 modelos con df_historico y devuelve, para cada uno,
    el ajuste sobre los años históricos y la proyección 2024-2100."""
    X = df_historico[["Year"]]
    y = df_historico[variable]

    # Modelo 1: regresión lineal
    modelo_lineal = LinearRegression()
    modelo_lineal.fit(X, y)

    # Modelo 2: regresión polinomial (grado 2)
    poly = PolynomialFeatures(degree=2, include_bias=False)
    X_poly = poly.fit_transform(X)
    modelo_polinomial = LinearRegression()
    modelo_polinomial.fit(X_poly, y)

    # Modelo 3: Random Forest
    modelo_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    modelo_rf.fit(X, y)

    return {
        "Regresión lineal": {
            "historico": modelo_lineal.predict(X),
            "futuro": modelo_lineal.predict(X_futuro),
        },
        "Regresión polinomial": {
            "historico": modelo_polinomial.predict(X_poly),
            "futuro": modelo_polinomial.predict(poly.transform(X_futuro)),
        },
        "Random Forest": {
            "historico": modelo_rf.predict(X),
            "futuro": modelo_rf.predict(X_futuro),
        },
    }


def _limite_superior(df_historico, df_proyectado, variable):
    # Evita que un modelo con valores extremos aplaste las demás curvas
    maximo = max(df_historico[variable].max(), df_proyectado[variable].max())
    return maximo * 1.5


def graficar_modelos_individuales(df_historico, df_proyectado, resultados,
                                  variable, etiqueta, lugar):
    """Una gráfica por modelo (ajuste histórico + proyección)."""
    techo = _limite_superior(df_historico, df_proyectado, variable)
    fig, ejes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

    for eje, (nombre, pred) in zip(ejes, resultados.items()):
        color = COLORES[nombre]
        eje.scatter(df_historico["Year"], df_historico[variable],
                    alpha=0.6, label="Datos históricos")
        eje.plot(df_historico["Year"], pred["historico"],
                 color=color, label=f"{nombre} (ajuste histórico)")
        eje.plot(anios_futuros.flatten(), pred["futuro"],
                 color=color, linestyle="--", label=f"{nombre} (proyección)")
        eje.set_title(nombre)
        eje.set_xlabel("Año")
        eje.ticklabel_format(style="plain", axis="y", useOffset=False)
        eje.set_ylim(0, techo)
        eje.legend(fontsize=8)
        eje.grid()

    ejes[0].set_ylabel(etiqueta)
    fig.suptitle(f"Proyección de {etiqueta.lower()} en {lugar}", fontsize=14)
    plt.tight_layout()
    plt.show()


def graficar_comparacion_owid(df_historico, df_proyectado, resultados,
                              variable, etiqueta, lugar):
    """Los 3 modelos + la proyección de OWID en una sola gráfica."""
    techo = _limite_superior(df_historico, df_proyectado, variable)
    plt.figure(figsize=(14, 7))

    plt.scatter(df_historico["Year"], df_historico[variable],
                label="Datos históricos", alpha=0.5)

    plt.plot(df_proyectado["Year"], df_proyectado[variable],
             color="black", linewidth=2.5, label="Proyección OWID")

    for nombre, pred in resultados.items():
        plt.plot(anios_futuros.flatten(), pred["futuro"],
                 color=COLORES[nombre], label=nombre)

    plt.title(f"Proyecciones de {etiqueta.lower()} en {lugar}: ML vs OWID")
    plt.xlabel("Año")
    plt.ylabel(etiqueta)
    plt.ylim(0, techo)
    plt.ticklabel_format(style="plain", axis="y", useOffset=False)
    plt.legend()
    plt.grid()
    plt.show()


def tabla_resumen(df_proyectado, resultados, variable, anios=(2030, 2050, 2075, 2100)):
    """Valores que predice cada modelo para algunos años."""
    indice = anios_futuros.flatten()
    tabla = pd.DataFrame(
        {"OWID": df_proyectado.set_index("Year")[variable].reindex(indice).to_numpy()},
        index=indice,
    )
    for nombre, pred in resultados.items():
        tabla[nombre] = pred["futuro"]
    tabla.index.name = "Año"
    return tabla.loc[list(anios)]

# EJERCICIO 1. China

## 1.1 Extraer el país

In [ ]:
pais = "China"

df_china = df[df["Entity"] == pais]

df_china.head()

In [ ]:
print("Rango de años:", df_china["Year"].min(), "-", df_china["Year"].max())
print("Número de filas:", df_china.shape[0])

## 1.2 Explorar visualmente

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(16, 5))

for eje, variable, titulo in zip(ejes, ["births", "deaths"], ["Nacimientos", "Defunciones"]):
    eje.plot(df_china["Year"], df_china[variable])
    eje.set_title(f"{titulo} en China")
    eje.set_xlabel("Año")
    eje.set_ylabel(titulo)
    eje.ticklabel_format(style="plain", axis="y", useOffset=False)
    eje.grid()

plt.tight_layout()
plt.show()

## 1.3 Separar histórico y proyectado (corte en 2023)

In [ ]:
df_historico_china = df_china[df_china["Year"] <= 2023]
df_proyectado_china = df_china[df_china["Year"] > 2023]

print("Histórico  ->", df_historico_china.shape, "| años:", df_historico_china["Year"].min(), "a", df_historico_china["Year"].max())
print("Proyectado ->", df_proyectado_china.shape, "| años:", df_proyectado_china["Year"].min(), "a", df_proyectado_china["Year"].max())

df_historico_china[["Year", "births", "deaths"]].isna().sum()

## 1.4 Nacimientos en China: entrenar los 3 modelos y graficar

In [ ]:
resultados_china_births = proyectar_tres_modelos(df_historico_china, "births")

graficar_modelos_individuales(
    df_historico_china, df_proyectado_china, resultados_china_births,
    "births", "Nacimientos", "China"
)

In [ ]:
graficar_comparacion_owid(
    df_historico_china, df_proyectado_china, resultados_china_births,
    "births", "Nacimientos", "China"
)

## 1.5 Defunciones en China: entrenar los 3 modelos y graficar

In [ ]:
resultados_china_deaths = proyectar_tres_modelos(df_historico_china, "deaths")

graficar_modelos_individuales(
    df_historico_china, df_proyectado_china, resultados_china_deaths,
    "deaths", "Defunciones", "China"
)

In [ ]:
graficar_comparacion_owid(
    df_historico_china, df_proyectado_china, resultados_china_deaths,
    "deaths", "Defunciones", "China"
)

## 1.6 Resumen numérico

In [ ]:
print("NACIMIENTOS – CHINA")
display(tabla_resumen(df_proyectado_china, resultados_china_births, "births").style.format("{:,.0f}"))

print("DEFUNCIONES – CHINA")
display(tabla_resumen(df_proyectado_china, resultados_china_deaths, "deaths").style.format("{:,.0f}"))

## 1.7 Observaciones (China)

Responde con lo que veas en tus gráficas:

1. ¿Qué modelo se parece más a la proyección de OWID en nacimientos? ¿Y en defunciones?
2. ¿Los nacimientos y las defunciones de China siguen una tendencia parecida entre sí? ¿Qué pasa con la regresión lineal y la polinomial en cada caso?
3. ¿Hay valores sin sentido (por ejemplo negativos) en alguna proyección? ¿A qué se debe?
4. ¿Qué hace Random Forest después de 2023?

*(Escribe tus observaciones aquí)*

# EJERCICIO 2. México a partir del año 2000

En la práctica original, los modelos se entrenaron con **todo** el histórico de México (1950–2023). Ahora solo se usan los datos **desde el año 2000**, es decir, 2000–2023 (24 observaciones). Las proyecciones siguen siendo 2024–2100.

## 2.1 Extraer México desde el 2000

In [ ]:
pais = "Mexico"

df_mexico = df[df["Entity"] == pais]

# Solo años a partir de 2000
df_mexico_2000 = df_mexico[df_mexico["Year"] >= 2000]

df_mexico_2000.head()

In [ ]:
print("Rango de años:", df_mexico_2000["Year"].min(), "-", df_mexico_2000["Year"].max())
print("Número de filas:", df_mexico_2000.shape[0])

## 2.2 Explorar visualmente

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(16, 5))

for eje, variable, titulo in zip(ejes, ["births", "deaths"], ["Nacimientos", "Defunciones"]):
    eje.plot(df_mexico_2000["Year"], df_mexico_2000[variable])
    eje.set_title(f"{titulo} en México (desde 2000)")
    eje.set_xlabel("Año")
    eje.set_ylabel(titulo)
    eje.ticklabel_format(style="plain", axis="y", useOffset=False)
    eje.grid()

plt.tight_layout()
plt.show()

## 2.3 Separar histórico y proyectado (corte en 2023)

In [ ]:
df_historico_mx2000 = df_mexico_2000[df_mexico_2000["Year"] <= 2023]
df_proyectado_mx2000 = df_mexico_2000[df_mexico_2000["Year"] > 2023]

print("Histórico  ->", df_historico_mx2000.shape, "| años:", df_historico_mx2000["Year"].min(), "a", df_historico_mx2000["Year"].max())
print("Proyectado ->", df_proyectado_mx2000.shape, "| años:", df_proyectado_mx2000["Year"].min(), "a", df_proyectado_mx2000["Year"].max())

df_historico_mx2000[["Year", "births", "deaths"]].isna().sum()

## 2.4 Nacimientos en México (desde 2000): entrenar los 3 modelos y graficar

In [ ]:
resultados_mx2000_births = proyectar_tres_modelos(df_historico_mx2000, "births")

graficar_modelos_individuales(
    df_historico_mx2000, df_proyectado_mx2000, resultados_mx2000_births,
    "births", "Nacimientos", "México (entrenado desde 2000)"
)

In [ ]:
graficar_comparacion_owid(
    df_historico_mx2000, df_proyectado_mx2000, resultados_mx2000_births,
    "births", "Nacimientos", "México (entrenado desde 2000)"
)

## 2.5 Defunciones en México (desde 2000): entrenar los 3 modelos y graficar

In [ ]:
resultados_mx2000_deaths = proyectar_tres_modelos(df_historico_mx2000, "deaths")

graficar_modelos_individuales(
    df_historico_mx2000, df_proyectado_mx2000, resultados_mx2000_deaths,
    "deaths", "Defunciones", "México (entrenado desde 2000)"
)

In [ ]:
graficar_comparacion_owid(
    df_historico_mx2000, df_proyectado_mx2000, resultados_mx2000_deaths,
    "deaths", "Defunciones", "México (entrenado desde 2000)"
)

## 2.6 Resumen numérico

In [ ]:
print("NACIMIENTOS – MÉXICO (entrenado desde 2000)")
display(tabla_resumen(df_proyectado_mx2000, resultados_mx2000_births, "births").style.format("{:,.0f}"))

print("DEFUNCIONES – MÉXICO (entrenado desde 2000)")
display(tabla_resumen(df_proyectado_mx2000, resultados_mx2000_deaths, "deaths").style.format("{:,.0f}"))

## 2.7 Extra: ¿cambia algo por entrenar solo desde 2000?

Comparamos la **regresión lineal** entrenada con todo el histórico (1950–2023) contra la entrenada solo con 2000–2023, junto con la proyección de OWID.

In [ ]:
df_historico_mx_completo = df_mexico[df_mexico["Year"] <= 2023]
df_proyectado_mx = df_mexico[df_mexico["Year"] > 2023]

fig, ejes = plt.subplots(1, 2, figsize=(16, 6))

for eje, variable, titulo in zip(ejes, ["births", "deaths"], ["Nacimientos", "Defunciones"]):
    res_completo = proyectar_tres_modelos(df_historico_mx_completo, variable)
    res_2000 = proyectar_tres_modelos(df_historico_mx2000, variable)

    eje.scatter(df_historico_mx_completo["Year"], df_historico_mx_completo[variable],
                alpha=0.4, label="Datos históricos")
    eje.plot(df_proyectado_mx["Year"], df_proyectado_mx[variable],
             color="black", linewidth=2.5, label="Proyección OWID")
    eje.plot(anios_futuros.flatten(), res_completo["Regresión lineal"]["futuro"],
             color="tab:blue", label="Lineal (1950–2023)")
    eje.plot(anios_futuros.flatten(), res_2000["Regresión lineal"]["futuro"],
             color="tab:orange", label="Lineal (2000–2023)")

    eje.set_title(f"{titulo} en México: ventana de entrenamiento")
    eje.set_xlabel("Año")
    eje.set_ylabel(titulo)
    eje.set_ylim(0, max(df_historico_mx_completo[variable].max(), df_proyectado_mx[variable].max()) * 1.5)
    eje.ticklabel_format(style="plain", axis="y", useOffset=False)
    eje.legend()
    eje.grid()

plt.tight_layout()
plt.show()

## 2.8 Observaciones (México desde 2000)

1. ¿Qué diferencias hay entre estas proyecciones y las de la práctica original (1950–2023)?
2. ¿Por qué cambia la regresión lineal al usar solo 2000–2023?
3. Con solo 24 observaciones, ¿la regresión polinomial se comporta mejor o peor? ¿Por qué?
4. ¿Random Forest cambió su comportamiento después de 2023?

*(Escribe tus observaciones aquí)*